In [1]:
# In this script we use gridded population layers (rasters at 30-m resolution)
# and aggregated pixels to HUC12 units
# so to each unit we assign total population 
# and demographic characteristics: mean share of white, mean share of black, etc.

In [2]:
import geopandas as gpd
import rasterio
from rasterstats import zonal_stats
from rasterstats.io import bounds_window

In [3]:
# Configurations

datadir = '/projects/standard/lenkne/oboiko/EJ/data/'
huc12_filepath = datadir + 'aoi_huc12_boundaries.gpkg'
var_filepath = datadir + 'processed/population/MissRivStates_gridded_pop_{}_{}.tif'
out_filepath = datadir + 'huc12_demographics.csv'
study_periods = ['2008_2012', '2013_2017', '2018_2022']
variables = [
    'total', 'share_nonhsp_white', 'share_black',
    'share_native', 'share_asian', 'share_hispanic',
    'share_below_poverty', 'share_2_below_poverty',
    'share_2_above_poverty'
]

In [4]:
# Load study area
huc12 = gpd.read_file(huc12_filepath)
huc12.head()

,huc12,name,areasqkm,states,Region,geometry
0,070300050401,Forest Lake-Sunrise River,43.43,MN,Gorge,"MULTIPOLYGON (((240228.337 2483439.303, 240271..."
1,070900070402,Fairfield Ditch Number 1-Green River,90.16,IL,Working River,"MULTIPOLYGON (((517568.047 2070542.219, 517539..."
2,070300030103,West Branch Kettle River,101.59,MN,Headwaters,"MULTIPOLYGON (((234614.153 2633575.283, 234693..."
3,070400080902,Crystal Creek,41.77,MN,Driftless,"MULTIPOLYGON (((362547.97 2317024.238, 362554...."
4,070400030605,Rose Valley,39.09,WI,Driftless,"MULTIPOLYGON (((332805.751 2374160.973, 332815..."


In [5]:
%%time
for study_period in study_periods:
    print ('\nProcessing', study_period)
    print ('Computing zonal population statistics')
    # define crs and window of raster files
    with rasterio.open(var_filepath.format(study_period, 'total')) as src:
        # this is defined to later read a subset of the raster file - it's faster than the entire raster
        window = bounds_window(huc12.total_bounds, src.transform)
        window_affine = src.window_transform(window)
    # process each variable
    for var in variables:
        print (var)
        src = rasterio.open(var_filepath.format(study_period, var))
        array = src.read(1, window=window)
        if var=='total':
            agg_stats = 'sum'
        else:
            agg_stats = 'mean'
        stats = zonal_stats(
            huc12, array, affine=window_affine, nodata=src.nodata, stats=[agg_stats]
        )
        huc12[f'{study_period}_{var}'] = [s[agg_stats] for s in stats]


Processing 2008_2012
Computing zonal population statistics
total
share_nonhsp_white
share_black
share_native
share_asian
share_hispanic
share_below_poverty
share_2_below_poverty
share_2_above_poverty

Processing 2013_2017
Computing zonal population statistics
total
share_nonhsp_white
share_black
share_native
share_asian
share_hispanic
share_below_poverty
share_2_below_poverty
share_2_above_poverty

Processing 2018_2022
Computing zonal population statistics
total
share_nonhsp_white
share_black
share_native
share_asian
share_hispanic
share_below_poverty
share_2_below_poverty
share_2_above_poverty
CPU times: user 20min 51s, sys: 5min 8s, total: 26min
Wall time: 30min 33s


In [7]:
# save results to a tabular format
# geometries can be dropped (will join to the boundaries file as needed)
# huc12.drop(columns=['geometry']).to_csv(out_filepath)